# ML-10 — Content Action Playbook

**Lane:** Refresh / Content Opportunity Scoring

This notebook converts the validated opportunity-scoring work into a practical, human-reviewed content action playbook.

The output is **decision support**, not an automated production system. Recommendations are ranked using observed signals and must be reviewed by a person before action.


## 1. Ranked actions + reason codes

The queue prioritizes pages using the baseline opportunity signals:

- **RC1:** High search volume
- **RC2:** Low impressions
- **RC3:** Low CTR
- **RC4:** Ranking position between 8 and 20
- **RC5:** Multiple positive signals combined

The score is a prioritization signal. It does not guarantee that refreshing a page will improve its future performance.


In [ ]:
import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

DATA_PATH = "YOUR_DATASET.csv"

# Change these only if your dataset uses different names.
URL_COLUMN = "url"
SEARCH_VOLUME = "search_volume"
IMPRESSIONS = "impressions_90d"
CTR = "ctr"
POSITION = "position"

# ------------------------------------------------------------
# Load data
# ------------------------------------------------------------

df = pd.read_csv(DATA_PATH)

required = [
    URL_COLUMN,
    SEARCH_VOLUME,
    IMPRESSIONS,
    CTR,
    POSITION
]

missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        "Update the configuration above."
    )

print("Dataset shape:", df.shape)
print("Required columns found.")


In [ ]:
# ------------------------------------------------------------
# Normalization helper
# ------------------------------------------------------------

def normalize(series):
    minimum = series.min()
    maximum = series.max()

    if maximum == minimum:
        return pd.Series(0.0, index=series.index)

    return (series - minimum) / (maximum - minimum)


df["sv_norm"] = normalize(df[SEARCH_VOLUME])
df["imp_norm"] = normalize(df[IMPRESSIONS])
df["ctr_norm"] = normalize(df[CTR])

# Position signal: positions 8–20 are treated as refresh opportunities.
df["position_score"] = (
    (df[POSITION] >= 8) &
    (df[POSITION] <= 20)
).astype(int)

# ------------------------------------------------------------
# Opportunity score
# ------------------------------------------------------------

df["opportunity_score"] = (
      0.40 * df["sv_norm"]
    + 0.30 * (1 - df["imp_norm"])
    + 0.20 * (1 - df["ctr_norm"])
    + 0.10 * df["position_score"]
)

print("Opportunity score calculated.")


In [ ]:
# ------------------------------------------------------------
# Reason codes
# ------------------------------------------------------------

def reason_codes(row):
    reasons = []

    if row["sv_norm"] > 0.70:
        reasons.append("RC1")

    if row["imp_norm"] < 0.30:
        reasons.append("RC2")

    if row["ctr_norm"] < 0.30:
        reasons.append("RC3")

    if 8 <= row[POSITION] <= 20:
        reasons.append("RC4")

    if len(reasons) >= 3:
        reasons.append("RC5")

    return ", ".join(reasons) if reasons else "None"


df["reason_codes"] = df.apply(reason_codes, axis=1)

# ------------------------------------------------------------
# Archetype
# ------------------------------------------------------------

def classify_archetype(row):
    if row["sv_norm"] > 0.70 and 8 <= row[POSITION] <= 20:
        return "High-demand mid-ranking"

    if row["imp_norm"] < 0.30 and row["ctr_norm"] < 0.30:
        return "Low-visibility / low-CTR"

    if row["ctr_norm"] < 0.30:
        return "CTR opportunity"

    if row["imp_norm"] < 0.30:
        return "Low-impression opportunity"

    if 8 <= row[POSITION] <= 20:
        return "Mid-ranking opportunity"

    return "Lower-priority"


df["archetype"] = df.apply(classify_archetype, axis=1)

print(df["archetype"].value_counts())


In [ ]:
# ------------------------------------------------------------
# Archetype -> action mapping
# ------------------------------------------------------------

action_map = {
    "High-demand mid-ranking":
        "Human review for content refresh and search-intent alignment",

    "Low-visibility / low-CTR":
        "Human review for title/snippet and content relevance",

    "CTR opportunity":
        "Human review of title, description, and search-intent match",

    "Low-impression opportunity":
        "Human review of indexing, relevance, and content coverage",

    "Mid-ranking opportunity":
        "Human review for targeted content refresh",

    "Lower-priority":
        "Monitor; no immediate refresh recommendation"
}

df["recommended_action"] = df["archetype"].map(action_map)

# Rank highest opportunity first.
baseline = df.sort_values(
    "opportunity_score",
    ascending=False
).reset_index(drop=True)

baseline["priority_rank"] = baseline.index + 1

print("Ranked queue created.")
display(
    baseline[
        [
            URL_COLUMN,
            "priority_rank",
            "opportunity_score",
            "reason_codes",
            "archetype",
            "recommended_action"
        ]
    ].head(20)
)


## 2. Intended use and limits

### Intended use

The playbook is intended for a content or SEO reviewer who needs a practical way to prioritize pages for manual review.

The queue answers:

> **Which pages show the strongest observed signals for a possible content refresh, and why?**

It does not answer:

> **Which pages will definitely improve if refreshed?**

### Limits

The score is based on observed historical signals. Search demand, impressions, CTR, and ranking position can change over time.

The recommendations can also be affected by seasonality, search-engine changes, competitors, technical issues, content quality, or planned editorial work.

The output should therefore be treated as **directional decision-support**, not a causal prediction or production automation system.


### Decay / refresh insight

Content opportunities can become stale as search demand, impressions, CTR, and rankings change.

A page that ranks highly in the queue today may not remain a high-priority page later. Conversely, a page that is currently low priority can become more relevant after its search environment changes.

The practical implication is to periodically regenerate the queue rather than treating one ranking as permanent.


In [ ]:
# Simple distribution check for the current queue.
print("Opportunity score summary:")
display(
    baseline["opportunity_score"].describe().to_frame()
)

print("\nTop archetypes:")
display(
    baseline["archetype"].value_counts().to_frame("count")
)


## 3. Human review + the no-go list

### Human review rules

Before acting on any recommendation, a reviewer should:

1. Check whether the page is already scheduled for an update.
2. Check whether the page has a clear search intent and relevant topic.
3. Review current ranking and traffic context.
4. Check whether the page has technical/indexing problems that content changes would not solve.
5. Review competitors or current search results where appropriate.
6. Confirm that the recommended change is proportionate to the opportunity.
7. Record the human decision and reason before making a change.

### What should NOT be automated

The system should **not** automatically:

- publish or rewrite content;
- delete pages;
- change canonical URLs;
- change internal-link structures without review;
- change metadata on every ranked page;
- claim that a refresh caused an outcome;
- override an editorial decision;
- make changes based only on the opportunity score.

The score should narrow the review queue, not replace the reviewer.


## Cost / value thinking

The queue is most useful when the expected value of reviewing a page is greater than the cost of the review.

A practical prioritization rule is:

**High opportunity + reasonable review cost → review sooner**

**Low opportunity + high review cost → monitor**

This is not a financial forecast. It is a decision-support principle for allocating limited content-review time.


In [ ]:
# Optional lightweight cost/value prioritization.
# These are planning categories, not financial predictions.

def review_priority(row):
    score = row["opportunity_score"]

    if score >= 0.70:
        return "High — review sooner"
    elif score >= 0.45:
        return "Medium — review when capacity allows"
    else:
        return "Low — monitor"

baseline["review_priority"] = baseline.apply(
    review_priority,
    axis=1
)

display(
    baseline[
        [
            URL_COLUMN,
            "priority_rank",
            "opportunity_score",
            "review_priority",
            "recommended_action"
        ]
    ].head(20)
)


## 4. Monitoring / retrain triggers

The playbook should be reviewed when the underlying search environment changes.

### Monitoring triggers

Regenerate or inspect the queue when:

- the data window is substantially newer;
- the distribution of search volume changes;
- impressions or CTR distributions shift;
- ranking-position distributions change;
- many previously high-priority pages no longer appear in the queue;
- important search-engine changes affect the environment.

### Retrain / re-evaluation triggers

The model should be re-evaluated when:

- measured performance drops materially on a newer validation period;
- feature distributions shift substantially;
- the target definition changes;
- new reliable features become available;
- the content-refresh workflow changes;
- observed errors become concentrated in a new type of page.

These triggers are monitoring rules, not claims that a specific threshold guarantees model failure.


In [ ]:
# Lightweight drift checks between the first and last halves of the data.
# This is a simple diagnostic, not a formal drift detector.

mid = len(df) // 2

early = df.iloc[:mid]
recent = df.iloc[mid:]

drift_rows = []

for col in [SEARCH_VOLUME, IMPRESSIONS, CTR, POSITION]:
    early_mean = pd.to_numeric(
        early[col], errors="coerce"
    ).mean()

    recent_mean = pd.to_numeric(
        recent[col], errors="coerce"
    ).mean()

    drift_rows.append({
        "feature": col,
        "early_mean": early_mean,
        "recent_mean": recent_mean,
        "absolute_change": abs(recent_mean - early_mean)
    })

drift_check = pd.DataFrame(drift_rows)

display(drift_check)

print(
    "\nUse this as a directional monitoring check; "
    "it is not a formal statistical drift test."
)


## 5. Exports for the paper

The paper will reuse the ranked queue generated here.

The queue is written to `work/outputs/`. The notebook regenerates the data file so that the underlying queue does not need to be committed to git.

A small metrics/summary JSON is also exported as a reproducible receipt.


In [ ]:
# ------------------------------------------------------------
# Export ranked queue
# ------------------------------------------------------------

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

queue_columns = [
    URL_COLUMN,
    "priority_rank",
    "opportunity_score",
    "reason_codes",
    "archetype",
    "recommended_action",
    "review_priority"
]

queue_path = "work/outputs/content_action_queue.csv"

baseline[queue_columns].to_csv(
    queue_path,
    index=False
)

print("Queue exported to:", queue_path)


In [ ]:
# ------------------------------------------------------------
# Export summary metrics / receipts
# ------------------------------------------------------------

summary = {
    "rows_scored": int(len(baseline)),
    "top_20_count": int(min(20, len(baseline))),
    "mean_opportunity_score": float(
        baseline["opportunity_score"].mean()
    ),
    "median_opportunity_score": float(
        baseline["opportunity_score"].median()
    ),
    "high_priority_count": int(
        (baseline["review_priority"] == "High — review sooner").sum()
    ),
    "note": (
        "Scores are observed, directional decision-support signals "
        "and do not guarantee refresh outcomes."
    )
}

metrics_path = "work/outputs/w07_playbook_summary.json"

with open(metrics_path, "w", encoding="utf-8") as f:
    import json
    json.dump(summary, f, indent=2)

print("Summary exported to:", metrics_path)
print(summary)


In [ ]:
# ------------------------------------------------------------
# Optional reusable figure
# ------------------------------------------------------------

import matplotlib.pyplot as plt

top_n = baseline.head(20).copy()

plt.figure(figsize=(10, 6))
plt.barh(
    top_n["priority_rank"].astype(str),
    top_n["opportunity_score"]
)
plt.xlabel("Opportunity Score")
plt.ylabel("Priority Rank")
plt.title("Top 20 Content Opportunity Scores")
plt.gca().invert_yaxis()
plt.tight_layout()

figure_path = "work/figures/w07_top20_opportunity_scores.png"
plt.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

print("Figure exported to:", figure_path)


## Self-check

- [x] Ranked actions and reason codes are included.
- [x] Archetype-to-action mapping is included.
- [x] Intended use and limits are stated.
- [x] Human-review rules are stated.
- [x] A no-go list identifies what should not be automated.
- [x] Cost/value thinking is included.
- [x] Monitoring and re-evaluation triggers are included.
- [x] The queue is exported to `work/outputs/`.
- [x] A summary receipt is exported.
- [x] A reusable figure is exported to `work/figures/`.
- [x] Claims use observed, measured, directional, and decision-support language.
- [ ] The notebook runs top-to-bottom after the dataset configuration is updated.
- [ ] The executed notebook is saved as `work/notebooks/w07_action_playbook.ipynb`.
- [ ] The notebook and permitted receipts/figures are committed to the repository.
